In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split


# -----------------------------
# Configuration
# -----------------------------
INPUT_FILE =  Path("data/processed/2026-07-27_preprocessed_bindingdb.csv")

OUTPUT_FILE = Path(
    "data/processed/"
    "bindingdb_subset_150k.csv"
)

# data =pd.read_csv(
#     INPUT_FILE,
#     nrows=5
# ).head()

# print(data)

# return

TARGET_COLUMN = "affinity"
SUBSET_SIZE = 150_000
NUMBER_OF_BINS = 20
RANDOM_STATE = 42

# Optional pKi range filtering.
# Set these to None to keep all finite pKi values.
MIN_PKI = None
MAX_PKI = None


# -----------------------------
# Load dataset
# -----------------------------
print("Loading dataset...")

df = pd.read_csv(
    INPUT_FILE,
    low_memory=False
)

print(f"Total records before checking: {len(df):,}")


# -----------------------------
# Validate target column
# -----------------------------
if TARGET_COLUMN not in df.columns:
    raise KeyError(
        f"Target column '{TARGET_COLUMN}' was not found.\n"
        f"Available columns: {list(df.columns)}"
    )


# -----------------------------
# Clean target values
# -----------------------------

# Convert pKi to numeric.
# Invalid strings will become NaN.
df[TARGET_COLUMN] = pd.to_numeric(
    df[TARGET_COLUMN],
    errors="coerce"
)

missing_count = df[TARGET_COLUMN].isna().sum()
positive_inf_count = np.isposinf(df[TARGET_COLUMN]).sum()
negative_inf_count = np.isneginf(df[TARGET_COLUMN]).sum()

print(f"Missing or non-numeric values: {missing_count:,}")
print(f"Positive infinity values: {positive_inf_count:,}")
print(f"Negative infinity values: {negative_inf_count:,}")

# Replace infinity values with NaN
df[TARGET_COLUMN] = df[TARGET_COLUMN].replace(
    [np.inf, -np.inf],
    np.nan
)

# Remove NaN and infinity values
df = df.dropna(
    subset=[TARGET_COLUMN]
).copy()


# -----------------------------
# Optional pKi range filtering
# -----------------------------
if MIN_PKI is not None:
    rows_before = len(df)
    df = df[df[TARGET_COLUMN] >= MIN_PKI].copy()

    print(
        f"Removed {rows_before - len(df):,} rows "
        f"below pKi {MIN_PKI}."
    )

if MAX_PKI is not None:
    rows_before = len(df)
    df = df[df[TARGET_COLUMN] <= MAX_PKI].copy()

    print(
        f"Removed {rows_before - len(df):,} rows "
        f"above pKi {MAX_PKI}."
    )

df = df.reset_index(drop=True)

print(f"Usable finite records: {len(df):,}")


# -----------------------------
# Check available records
# -----------------------------
if len(df) < SUBSET_SIZE:
    raise ValueError(
        f"Only {len(df):,} valid records are available. "
        f"Cannot create a subset of {SUBSET_SIZE:,}."
    )


# -----------------------------
# Create affinity-value bins
# -----------------------------
df["_affinity_bin"] = pd.qcut(
    df[TARGET_COLUMN],
    q=NUMBER_OF_BINS,
    labels=False,
    duplicates="drop"
)

# Remove rows that were not assigned to a bin
df = df.dropna(
    subset=["_affinity_bin"]
).copy()

df["_affinity_bin"] = df["_affinity_bin"].astype(int)

number_of_created_bins = df["_affinity_bin"].nunique()

print(f"Affinity bins created: {number_of_created_bins}")

print("\nRecords per affinity bin:")
print(
    df["_affinity_bin"]
    .value_counts()
    .sort_index()
)


# -----------------------------
# Check stratification feasibility
# -----------------------------
minimum_bin_size = df["_affinity_bin"].value_counts().min()

if minimum_bin_size < 2:
    raise ValueError(
        "At least one affinity bin contains fewer than two records. "
        "Reduce NUMBER_OF_BINS."
    )


# -----------------------------
# Select exactly 100,000 records
# -----------------------------
subset_df, _ = train_test_split(
    df,
    train_size=SUBSET_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["_affinity_bin"]
)


# -----------------------------
# Remove temporary column
# -----------------------------
subset_df = subset_df.drop(
    columns=["_affinity_bin"]
).reset_index(drop=True)


# -----------------------------
# Final validation
# -----------------------------
if len(subset_df) != SUBSET_SIZE:
    raise ValueError(
        f"Expected {SUBSET_SIZE:,} rows, "
        f"but generated {len(subset_df):,} rows."
    )

if subset_df[TARGET_COLUMN].isna().any():
    raise ValueError(
        "The generated subset contains missing target values."
    )

if not np.isfinite(subset_df[TARGET_COLUMN]).all():
    raise ValueError(
        "The generated subset contains infinity values."
    )


# -----------------------------
# Save subset
# -----------------------------
OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

subset_df.to_csv(
    OUTPUT_FILE,
    index=False
)


# -----------------------------
# Display results
# -----------------------------
print("\nSubset successfully created.")
print(f"Subset records: {len(subset_df):,}")
print(f"Subset saved to: {OUTPUT_FILE.resolve()}")

print("\nAffinity distribution:")
print(subset_df[TARGET_COLUMN].describe())

print("\nFinal target-value checks:")
print(
    f"NaN values: "
    f"{subset_df[TARGET_COLUMN].isna().sum():,}"
)
print(
    f"Positive infinity values: "
    f"{np.isposinf(subset_df[TARGET_COLUMN]).sum():,}"
)
print(
    f"Negative infinity values: "
    f"{np.isneginf(subset_df[TARGET_COLUMN]).sum():,}"
)

Loading dataset...
Total records before checking: 1,989,115
Missing or non-numeric values: 0
Positive infinity values: 0
Negative infinity values: 0
Usable finite records: 1,989,115
Affinity bins created: 20

Records per affinity bin:
_affinity_bin
0      99457
1      99455
2     100150
3      98796
4      99425
5      99490
6     122456
7      76420
8     100221
9      98736
10    114838
11     85203
12    100945
13     97957
14     99695
15    102240
16     95432
17    100428
18     98487
19     99284
Name: count, dtype: int64

Subset successfully created.
Subset records: 150,000
Subset saved to: /Volumes/MySSD/python/Drug-target-interaction-using-machine-learning/data/processed/bindingdb_subset_150k.csv

Affinity distribution:
count    150000.000000
mean          6.785627
std           1.433964
min          -0.845098
25%           5.780193
50%           6.863279
75%           7.795880
max          13.267606
Name: affinity, dtype: float64

Final target-value checks:
NaN values: 0
Pos